In [1]:
%%capture
!pip install langchain_community
!pip install langchain_openai
!pip install langchain_huggingface
!pip install pypdf
!pip install faiss-cpu
!pip install langchain_classic

In [2]:
%%capture
!pip install langchain_text_splitters

In [3]:
pdf_path = "/content/human-nutrition-text.pdf"

In [4]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_3791/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

In [6]:
loader = PyPDFLoader(
    pdf_path
)

In [7]:
documents = loader.load()

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [9]:
chunks = splitter.split_documents(
    documents
)

In [10]:
chunks[0].page_content

'Human Nutrition: 2020 Edition'

In [11]:
len(chunks)

3577

In [12]:
chunks[38].page_content

'University of Hawai‘i at Mānoa Food Science and \nHuman Nutrition Program and Human Nutrition \nProgram \n760 \nUnderstanding the Bigger Picture of Dietary \nGuidelines \nUniversity of Hawai‘i at Mānoa Food Science and \nHuman Nutrition Program and Human Nutrition \nProgram \n768 \nPart XIII. Chapter 13. Lifespan Nutrition From \nPregnancy to the Toddler Years \nIntroduction \nUniversity of Hawai‘i at Mānoa Food Science and \nHuman Nutrition Program and Human Nutrition \nProgram \n779 \nPregnancy'

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model="sentence-transformers/all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
from langchain_community.vectorstores import FAISS

In [15]:
vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

In [29]:
vectorstore.save_local("nutrition_index")

In [16]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs = {'k':3}
)

In [17]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [18]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo")

In [19]:
from langchain_core.prompts import ChatPromptTemplate

In [20]:
system_prompt = (
    "You are a passionate Pokémon expert with extensive knowledge of the Pokémon universe, including games, anime, manga, movies, characters, regions, lore, and unreleased content such as the cancelled 'Team Rocket vs Team Plasma' episode/movie.\n"
    "Answer questions accurately using the provided context and your Pokémon knowledge. If you are unsure or the information is not available, simply say 'I don't know'.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [21]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [22]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [23]:
question_answer_chain = create_stuff_documents_chain(llm,prompt) | parser


In [24]:
from langchain_classic.chains import create_retrieval_chain


In [25]:
rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [26]:
def ask_question(query):
    response = rag_chain.invoke({'input':query})
    return response['answer']


In [28]:
ask_question("Is intermittent fasting effective for weight loss?")

"Intermittent fasting has gained popularity for weight loss, and some studies suggest it can be effective for some individuals. It is a pattern of eating that alternates between fasting and eating periods. During the fasting periods, the body utilizes stored fat for energy, which can lead to weight loss. However, the effectiveness of intermittent fasting can vary from person to person, and not all individuals may find it sustainable in the long term.\n\nIt's essential to note that successful weight loss is a multifaceted journey that involves not only diet but also physical activity, consistency, and overall lifestyle changes. For personalized advice on weight loss strategies, consulting with a healthcare provider or a nutritionist would be beneficial."

In [30]:
!zip -r my_folder.zip /content/nutrition_index

  adding: content/nutrition_index/ (stored 0%)
  adding: content/nutrition_index/index.pkl (deflated 66%)
  adding: content/nutrition_index/index.faiss (deflated 7%)
